# SODAS on Google Colab (T4)

Run top to bottom. Runtime -> Change runtime type -> T4 GPU.
Colab's local disk resets with the runtime, so outputs (`runs/`, `results/`) are symlinked to Google Drive. Re-run sections 1 to 3 after every reset, then continue from where you stopped.

## 1. GPU, Drive, repo

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from getpass import getpass
if not os.path.exists('/content/SODAS'):
    token = getpass('GitHub token (repo read access): ')
    !git clone https://{token}@github.com/AnhadKohli123/SODAS.git /content/SODAS
%cd /content/SODAS
!git pull
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 2. Keep outputs on Drive

In [ ]:
%%bash
cd /content/SODAS
mkdir -p /content/drive/MyDrive/SODAS_out/runs /content/drive/MyDrive/SODAS_out/results
rm -rf runs results
ln -sfn /content/drive/MyDrive/SODAS_out/runs runs
ln -sfn /content/drive/MyDrive/SODAS_out/results results
ls -la runs results | head

## 3. Data (about 2 GB, re-download after each runtime reset)

In [ ]:
import os
os.chdir('/content/SODAS')
for p in ['data/coco_json', 'data/yolo', 'data/raw']:
    if os.path.lexists(p) and not os.path.isdir(p):
        os.remove(p)
        print('removed stray file:', p)

In [ ]:
%%bash
cd /content/SODAS
bash scripts/download_visdrone.sh
python scripts/convert_visdrone.py

In [ ]:
%%bash
cd /content/SODAS
sed -i 's#^path: .*#path: /content/SODAS/data/yolo#' configs/data_visdrone.yaml
head -3 configs/data_visdrone.yaml
ls data/yolo
ls data/yolo/val/images | head -3

## 4. Smoke test (about 10 min)
Trains 1 epoch, then checks that identity tiling reproduces standard inference. Do not continue until this passes.

In [ ]:
%%bash
cd /content/SODAS
sed -i 's/^epochs: .*/epochs: 1/' configs/train_640.yaml
rm -rf runs/y8s_640_s0
python scripts/train.py --cfg configs/train_640.yaml --seed 0
ls runs/y8s_640_s0/weights

In [ ]:
import os
os.chdir('/content/SODAS')
os.environ['SODAS_MODEL'] = 'runs/y8s_640_s0/weights/best.pt'
os.environ['SODAS_GT'] = 'data/coco_json/val.json'
os.environ['SODAS_IMG'] = 'data/yolo/val/images'
!pytest -q

In [ ]:
%%bash
cd /content/SODAS
sed -i 's/^epochs: .*/epochs: 100/' configs/train_640.yaml
grep epochs configs/train_640.yaml
rm -rf runs/y8s_640_s0

## 5. Full training
One cell per run. Each is long, so run one or two per Colab session.
If a session dies mid-run, resume with the last cell of this section.

In [ ]:
%%bash
cd /content/SODAS
python scripts/train.py --cfg configs/train_640.yaml --seed 0

In [ ]:
%%bash
cd /content/SODAS
python scripts/train.py --cfg configs/train_640.yaml --seed 1

In [ ]:
%%bash
cd /content/SODAS
python scripts/train.py --cfg configs/train_960.yaml --seed 0

In [ ]:
%%bash
cd /content/SODAS
python scripts/train.py --cfg configs/train_960.yaml --seed 1

In [ ]:
!yolo train resume model=/content/SODAS/runs/y8s_640_s0/weights/last.pt

## 6. Evaluate on val
Baselines and the tiled grid. The grid skips finished configs, so it is safe to re-run after a disconnect.

In [ ]:
%%bash
cd /content/SODAS
for res in 640 960; do
  for seed in 0 1; do
    python scripts/run_baselines.py --res $res --seed $seed
    python scripts/run_sahi_grid.py --res $res --seed $seed
  done
done

In [ ]:
%%bash
cd /content/SODAS
for res in 640 960; do
  for seed in 0 1; do
    python scripts/run_sahi_grid.py --res $res --seed $seed --ablation
  done
done

## 7. Select the frozen config, then score test-dev once

In [ ]:
%%bash
cd /content/SODAS
python scripts/select_config.py
cat results/frozen.json

In [ ]:
%%bash
cd /content/SODAS
python scripts/run_test_frozen.py

## 8. Figures

In [ ]:
%%bash
cd /content/SODAS
mkdir -p figures
python figures/make_scatter.py
python figures/make_heatmap.py --res 640 --full 0
python figures/make_heatmap.py --res 960 --full 0

In [ ]:
from IPython.display import Image, display
display(Image('/content/SODAS/figures/scatter_latency_aps.png'))

In [ ]:
!ls /content/SODAS/results/test/dets | head -20

In [ ]:
!cp -r /content/SODAS/figures /content/drive/MyDrive/SODAS_out/figures